# 1. Build the COCO benchmark

Builds a frozen benchmark of **degraded/clean pairs** from COCO val2017, images never used in training:
- one 256×256 crop per image;
- BOPBTL's synthetic degradation, with a fixed seed per pair;
- 50% grayscale;
- stored as PNG pairs on Drive.

Every model is then scored on exactly the same pairs. This notebook also scores the reference rows: the degraded input, BOPBTL's mapping and the **VAE ceiling** (the clean image passed through the frozen VAE, i.e. a perfect mapping).

**Time:** download ~3 min, building ~10–15 min (once; resumable), evaluation ~3–5 min per row.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
if not os.path.isdir("/content/Bringing-Old-Photos-Back-to-Life"):
    subprocess.run(["git", "clone", "-q", "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life",
                    "/content/Bringing-Old-Photos-Back-to-Life"], check=True)
sys.path.insert(0, "/content")
!pip install -q lpips pytorch-fid pyiqa

## Code (identical to `src/`)

### `old_photo_model.py`

In [ ]:
%%writefile /content/old_photo_model.py
"""
OldPhotoModel: single-class wrapper around microsoft/Bringing-Old-Photos-Back-to-Life
(Global restoration stage). Finds the repo (Colab: /content/...), downloads the pretrained
weights if missing, builds VAE A, VAE B and the mapping network, loads their weights and
verifies that every checkpoint key matched.

    from old_photo_model import OldPhotoModel
    obj = OldPhotoModel(mode="quality", gpu=0)

    x   = obj.to_tensor(PIL_image)
    z_A = obj.encode_A(x)
    z_B = obj.map(z_A)
    y   = obj.decode_B(z_B)            # or obj.translate(x)
    obj.to_image(y).save("out.png")
    obj.restore(PIL_image).save("out.png")

Modes: "quality", "scratch" (needs mask), "scratch_hr" (needs mask, high-res).
Scratch modes: x, m = obj.prepare(image, mask); obj.map(z_A, m) / obj.translate(x, m).
"""
import contextlib
import glob
import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image, ImageFilter

_REPO_ZIP = "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/archive/refs/heads/master.zip"
_CKPT_ZIP = ("https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/"
             "releases/download/v1.0/global_checkpoints.zip")
_DEFAULT_ROOT = "/content/Bringing-Old-Photos-Back-to-Life"

# (checkpoint sub-folder, network label) required per mode -> files named "<epoch>_net_<label>.pth"
_REQUIRED = {
    "quality":    [("VAE_A_quality", "G"), ("VAE_B_quality", "G"), ("mapping_quality", "mapping_net")],
    "scratch":    [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_scratch", "mapping_net")],
    "scratch_hr": [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_Patch_Attention", "mapping_net")],
}


def _download(url, dest):
    print(f"[OldPhotoModel] downloading {url}")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)


def _find_repo(hint=None):
    """Return repo root: hint, /content/<repo>, or any folder under /content containing Global/models/mapping_model.py."""
    marker = os.path.join("Global", "models", "mapping_model.py")
    cands = [hint, _DEFAULT_ROOT, _DEFAULT_ROOT + "-master"]
    cands += glob.glob("/content/*/") + glob.glob("/content/*/*/")
    for c in cands:
        if c and os.path.isfile(os.path.join(c, marker)):
            return os.path.abspath(c)
    return None


@contextlib.contextmanager
def _cpu_safe_torch_load():
    """The repo calls torch.load(path) without map_location, which fails on CPU-only machines."""
    orig = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault("map_location", "cpu")
        return orig(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = orig


class OldPhotoModel:

    def __init__(self, repo_root=None, mode="quality", gpu=0, auto_download=True, verify=True,
                 test_mode="Full"):
        assert mode in _REQUIRED, f"mode must be one of {list(_REQUIRED)}"
        assert test_mode in ("Full", "Scale"), "test_mode: 'Full' (keep size) or 'Scale' (short side -> 256)"
        self.mode = mode
        self.test_mode = test_mode  # only used in quality mode, like --test_mode in Global/test.py

        found = _find_repo(repo_root)
        if found is None:
            if not auto_download:
                raise FileNotFoundError("Repo not found under /content; pass repo_root=...")
            repo_root = os.path.abspath(repo_root or _DEFAULT_ROOT)
            self._ensure_repo(repo_root)  # zip download as fallback
        else:
            repo_root = found
        global_dir = os.path.join(repo_root, "Global")
        print(f"[OldPhotoModel] using repo at {repo_root}")

        if auto_download:
            self._ensure_weights(global_dir, mode)
        else:
            self._check_weights(global_dir, mode, raise_error=True)

        if gpu >= 0 and not torch.cuda.is_available():
            print("[OldPhotoModel] CUDA not available -> running on CPU")
            gpu = -1

        opt = self._build_opt(global_dir, mode, gpu)

        from models.mapping_model import Pix2PixHDModel_Mapping  # repo code

        prev = os.getcwd()
        os.chdir(global_dir)
        try:
            with _cpu_safe_torch_load():
                core = Pix2PixHDModel_Mapping()
                core.initialize(opt)  # builds netG_A, netG_B, mapping_net + loads weights
        finally:
            os.chdir(prev)

        self.vae_A = core.netG_A      # old-photo domain VAE
        self.vae_B = core.netG_B      # clean domain VAE
        self.mapping = core.mapping_net
        for net in (self.vae_A, self.vae_B, self.mapping):
            net.eval()
            for p in net.parameters():
                p.requires_grad = False

        self.uses_mask = bool(getattr(opt, "NL_use_mask", False))
        self._hr = bool(getattr(opt, "inference_optimize", False))
        self._mask_dilation = int(getattr(opt, "mask_dilation", 0) or 0)
        self.device = next(self.mapping.parameters()).device
        self._core = core
        self._opt = opt

        if verify:
            self._verify_weights(opt, mode)

    # ------------------------------------------------------------------ #
    # Core methods
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def encode_A(self, x):
        return self.vae_A.forward(x, flow="enc")

    @torch.no_grad()
    def encode_B(self, x):
        return self.vae_B.forward(x, flow="enc")

    @torch.no_grad()
    def map(self, z_A, mask=None):
        if not self.uses_mask:
            return self.mapping(z_A)
        if mask is None:
            raise ValueError("scratch mode: pass mask (1,1,H,W) tensor, same H/W as the input image "
                             "(use obj.prepare(image, mask)).")
        if self._hr:
            return self.mapping.inference_forward(z_A, mask)
        return self.mapping(z_A, mask)

    @torch.no_grad()
    def decode_A(self, z_A):
        return self.vae_A.forward(z_A, flow="dec")

    @torch.no_grad()
    def decode_B(self, z_B):
        return self.vae_B.forward(z_B, flow="dec")

    @torch.no_grad()
    def translate(self, x_A, mask=None):
        z_A = self.encode_A(x_A)
        z_B = self.map(z_A, mask)
        return self.decode_B(z_B)

    # ------------------------------------------------------------------ #
    # Helpers: PIL <-> tensor, and one-call restore
    # ------------------------------------------------------------------ #
    def _fit(self, img):
        """Same resizing as Global/test.py (BILINEAR, multiples of 4).
        Scratch modes never rescale in test.py, so only the multiple-of-4 rounding applies there."""
        w, h = img.size
        ow, oh = w, h
        if self.test_mode == "Scale" and not self.uses_mask:
            if w < h:
                ow, oh = 256, h / w * 256
            else:
                oh, ow = 256, w / h * 256
        size = (int(round(ow / 4) * 4), int(round(oh / 4) * 4))
        return img if size == (w, h) else img.resize(size, Image.BILINEAR)

    def to_tensor(self, image):
        """PIL -> (1,3,H,W) tensor in [-1,1]; H,W rounded to multiples of 4."""
        image = self._fit(image.convert("RGB"))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return ((x - 0.5) / 0.5).unsqueeze(0).to(self.device)

    def mask_to_tensor(self, mask, size):
        """PIL mask (white = scratch) -> (1,1,H,W) tensor, resized to size=(W,H)."""
        m = mask.convert("RGB").resize(size, Image.NEAREST)
        m = torch.from_numpy(np.asarray(m, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return m[:1].unsqueeze(0).to(self.device)

    def prepare(self, image, mask=None):
        """PIL (+ PIL mask in scratch modes) -> (x, mask_tensor or None), preprocessed like Global/test.py:
        size handled like test.py, mask dilated (HR mode), scratched pixels painted white."""
        img = self._fit(image.convert("RGB"))
        if not self.uses_mask:
            return self.to_tensor(img), None
        if mask is None:
            raise ValueError("scratch mode needs a mask image (white = scratch).")
        mk = mask.convert("RGB").resize(img.size, Image.NEAREST)
        for _ in range(self._mask_dilation):
            mk = mk.filter(ImageFilter.MaxFilter(3))
        m = np.asarray(mk, dtype=np.float32) / 255.0
        holed = np.asarray(img, dtype=np.float32) * (1.0 - m) + 255.0 * m
        img = Image.fromarray(holed.round().clip(0, 255).astype(np.uint8))
        return self.to_tensor(img), self.mask_to_tensor(mk, img.size)

    @staticmethod
    def to_image(x, normalize=True):
        """(1,3,H,W) tensor in [-1,1] -> PIL. normalize=True min-max stretches like
        save_image(..., normalize=True) in Global/test.py; False = plain clamp."""
        y = (x.detach().cpu().float() + 1.0) / 2.0
        if normalize:
            lo, hi = y.min(), y.max()
            y = (y - lo) / (hi - lo).clamp_min(1e-5)
        y = y.clamp(0, 1)[0]
        return Image.fromarray((y.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

    def restore(self, image, mask=None, keep_size=True, normalize=True):
        """PIL in -> PIL out (full pipeline)."""
        x, m = self.prepare(image, mask)
        out = self.to_image(self.translate(x, m), normalize)
        if keep_size and out.size != image.size:
            out = out.resize(image.size, Image.BICUBIC)
        return out

    # ------------------------------------------------------------------ #
    # Setup: repo, weights, options
    # ------------------------------------------------------------------ #
    @staticmethod
    def _ensure_repo(repo_root):
        if os.path.isfile(os.path.join(repo_root, "Global", "models", "mapping_model.py")):
            return
        os.makedirs(os.path.dirname(repo_root), exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "repo.zip")
            _download(_REPO_ZIP, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(tmp)
            extracted = os.path.join(tmp, "Bringing-Old-Photos-Back-to-Life-master")
            if os.path.exists(repo_root):
                shutil.rmtree(repo_root)
            shutil.move(extracted, repo_root)

    def _ensure_weights(self, global_dir, mode):
        """Download the checkpoint zip (layout: checkpoints/restoration/<folder>/*.pth) and copy
        any missing or empty files for the needed folders."""
        if not self._missing(global_dir, mode):
            return
        target = os.path.join(global_dir, "checkpoints", "restoration")
        wanted = {f for f, _ in _REQUIRED[mode]}
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "global_checkpoints.zip")
            _download(_CKPT_ZIP, zpath)
            extract_dir = os.path.join(tmp, "extracted")
            with zipfile.ZipFile(zpath) as z:
                z.extractall(extract_dir)
            src_root = os.path.join(extract_dir, "checkpoints", "restoration")
            for folder in wanted:
                os.makedirs(os.path.join(target, folder), exist_ok=True)
                for f in os.listdir(os.path.join(src_root, folder)):
                    src, dst = os.path.join(src_root, folder, f), os.path.join(target, folder, f)
                    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
                        shutil.copy2(src, dst)
        self._check_weights(global_dir, mode, raise_error=True)

    @staticmethod
    def _missing(global_dir, mode):
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        missing = []
        for folder, label in _REQUIRED[mode]:
            if not glob.glob(os.path.join(ckpt, folder, f"*_net_{label}.pth")):
                missing.append(os.path.join(ckpt, folder, f"latest_net_{label}.pth"))
        return missing

    def _check_weights(self, global_dir, mode, raise_error):
        missing = self._missing(global_dir, mode)
        if missing and raise_error:
            raise FileNotFoundError("Missing checkpoints:\n  " + "\n  ".join(missing) +
                                    f"\nDownload {_CKPT_ZIP} and unzip it inside {global_dir}")
        return missing

    @staticmethod
    def _build_opt(global_dir, mode, gpu):
        if global_dir not in sys.path:
            sys.path.insert(0, global_dir)  # repo imports `options`, `models`, `util` from Global/
        from options.test_options import TestOptions

        flag = "--Quality_restore" if mode == "quality" else "--Scratch_and_Quality_restore"
        backup = sys.argv
        sys.argv = ["old_photo_model", flag, "--gpu_ids", str(gpu)]
        if mode == "scratch_hr":
            sys.argv.append("--HR")
        try:
            opt = TestOptions().parse(save=False)
        finally:
            sys.argv = backup

        # mirrors parameter_set() in Global/test.py
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        opt.serial_batches = True
        opt.no_flip = True
        opt.label_nc = 0
        opt.n_downsample_global = 3
        opt.mc = 64
        opt.k_size = 4
        opt.start_r = 1
        opt.mapping_n_block = 6
        opt.map_mc = 512
        opt.no_instance = True
        opt.checkpoints_dir = ckpt

        if mode == "quality":
            opt.name = "mapping_quality"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_quality")
        else:
            opt.NL_res = True
            opt.use_SN = True
            opt.correlation_renormalize = True
            opt.NL_use_mask = True
            opt.NL_fusion_method = "combine"
            opt.non_local = "Setting_42"
            opt.name = "mapping_scratch"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_scratch")
            if mode == "scratch_hr":
                opt.HR = True
                opt.mapping_exp = 1              # Mapping_Model_with_mask_2 (patch attention)
                opt.inference_optimize = True    # enables mapping.inference_forward
                opt.mask_dilation = 3
                opt.name = "mapping_Patch_Attention"
        return opt

    def _verify_weights(self, opt, mode):
        """Re-load each checkpoint with strict=True so a silent partial load can't go unnoticed."""
        ckpt = opt.checkpoints_dir
        vae_epoch = getattr(opt, "use_vae_which_epoch", "latest")
        map_epoch = getattr(opt, "which_epoch", "latest")
        jobs = [
            (self.vae_A, os.path.join(opt.load_pretrainA, f"{vae_epoch}_net_G.pth")),
            (self.vae_B, os.path.join(opt.load_pretrainB, f"{vae_epoch}_net_G.pth")),
            (self.mapping, os.path.join(ckpt, opt.name, f"{map_epoch}_net_mapping_net.pth")),
        ]
        for net, path in jobs:
            try:
                net.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
            except Exception as e:  # noqa: BLE001
                raise RuntimeError(
                    f"Checkpoint {path} does not match the network built for mode '{mode}': {e}\n"
                    "The option flags in _build_opt() probably differ from Global/test.py."
                ) from e

### `degradation.py`

In [ ]:
%%writefile /content/degradation.py
"""
Port of the synthetic degradation pipeline of microsoft/Bringing-Old-Photos-Back-to-Life
(Global/data/online_dataset_for_old_photos.py): blur, Gaussian / speckle / salt-and-pepper noise,
downscaling and JPEG compression, applied in random order.
"""
import random
from io import BytesIO

import cv2
import numpy as np
from PIL import Image


def _pil_to_np(img):
    ar = np.array(img)
    ar = ar.transpose(2, 0, 1) if ar.ndim == 3 else ar[None, ...]
    return ar.astype(np.float32) / 255.0


def _np_to_pil(ar):
    ar = np.clip(ar * 255, 0, 255).astype(np.uint8)
    ar = ar[0] if ar.shape[0] == 1 else ar.transpose(1, 2, 0)
    return Image.fromarray(ar)


def synthesize_salt_pepper(image, amount, salt_vs_pepper):
    img = _pil_to_np(image)
    out = img.copy()
    flipped = np.random.choice([True, False], size=img.shape, p=[amount, 1 - amount])
    salted = np.random.choice([True, False], size=img.shape, p=[salt_vs_pepper, 1 - salt_vs_pepper])
    peppered = ~salted
    out[flipped & salted] = 1
    out[flipped & peppered] = 0.0
    return _np_to_pil(np.clip(out, 0, 1).astype(np.float32))


def synthesize_gaussian(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape)
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_speckle(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape) * img
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_low_resolution(img):
    w, h = img.size
    new_w = random.randint(int(w / 2), w)
    new_h = random.randint(int(h / 2), h)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    method = Image.NEAREST if random.uniform(0, 1) < 0.5 else Image.BILINEAR
    return img.resize((w, h), method)


def convert_to_jpeg(img, quality):
    with BytesIO() as f:
        img.save(f, format="JPEG", quality=quality)
        f.seek(0)
        return Image.open(f).convert("RGB")


def blur_image_v2(img):
    x = np.array(img)
    kernel_size = random.choice([(3, 3), (5, 5), (7, 7)])
    std = random.uniform(1.0, 5.0)
    blur = cv2.GaussianBlur(x, kernel_size, std)
    return Image.fromarray(blur.astype(np.uint8))


def online_add_degradation_v2(img):
    """The repo's exact composite degradation used for training domain A / the mapping net."""
    task_id = np.random.permutation(4)
    for x in task_id:
        if x == 0 and random.uniform(0, 1) < 0.7:
            img = blur_image_v2(img)
        if x == 1 and random.uniform(0, 1) < 0.7:
            flag = random.choice([1, 2, 3])
            if flag == 1:
                img = synthesize_gaussian(img, 5, 50)
            elif flag == 2:
                img = synthesize_speckle(img, 5, 50)
            elif flag == 3:
                img = synthesize_salt_pepper(img, random.uniform(0, 0.01), random.uniform(0.3, 0.8))
        if x == 2 and random.uniform(0, 1) < 0.7:
            img = synthesize_low_resolution(img)
        if x == 3 and random.uniform(0, 1) < 0.7:
            img = convert_to_jpeg(img, random.randint(40, 100))
    return img


def irregular_hole_synthesize(img, mask):
    """Paint the masked (white) region white, exactly as the repo does before encoding domain A in scratch mode."""
    img_np = np.array(img).astype("uint8")
    mask_np = np.array(mask).astype("uint8") / 255
    img_new = img_np * (1 - mask_np) + mask_np * 255
    hole_img = Image.fromarray(img_new.astype("uint8")).convert("RGB")
    return hole_img, mask.convert("L")

### `online_dataset.py`

In [ ]:
%%writefile /content/online_dataset.py
"""
Training pairs generated on the fly: every __getitem__ draws a fresh clean image, a fresh random crop and a
fresh random degradation, as in the original BOPBTL training, so the model never sees the same degraded
instance twice.

Validation uses a fixed, in-memory set built once with a fixed seed from held-out source images (never used
for training), so validation numbers are comparable across runs.
"""
import glob
import math
import os
import random

import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from degradation import online_add_degradation_v2

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".ppm")
FINE_SIZE = 256

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # -> [-1,1]
])


def list_clean_images(clean_dir, min_side=256):
    files = sorted(f for f in glob.glob(os.path.join(clean_dir, "*")) if f.lower().endswith(IMG_EXT))
    kept, corrupt = [], 0
    for f in files:
        try:
            with Image.open(f) as im:
                im.verify()          # catches truncated/corrupt files
            with Image.open(f) as im:  # verify() invalidates the handle, reopen to read size
                w, h = im.size
        except Exception:
            corrupt += 1
            continue
        if w >= min_side and h >= min_side:
            kept.append(f)
    print(f"{len(kept)}/{len(files)} images pass the >= {min_side}px filter "
          f"({corrupt} unreadable/corrupt files skipped)")
    return kept


def _crop_params(size, fine_size=FINE_SIZE):
    w, h = size
    x = random.randint(0, max(0, w - fine_size))
    y = random.randint(0, max(0, h - fine_size))
    flip = random.random() > 0.5
    return (x, y), flip


def _apply_crop_flip(img, pos, flip, fine_size=FINE_SIZE):
    ow, oh = img.size
    x1, y1 = pos
    if ow > fine_size or oh > fine_size:
        img = img.crop((x1, y1, x1 + fine_size, y1 + fine_size))
    if flip:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    return img


def make_online_pair(clean_path, grayscale_prob=0.1):
    """Fresh (A, B) tensor pair every call: new crop position, new flip, new
    degradation realization -- even for the same source image.

    grayscale_prob: fraction of pairs where BOTH A and B are desaturated
    (matches the repo's own training, which also keeps B grayscale when A is).
    Real target old photos are overwhelmingly B&W, so the default 0.1 (matching
    the original repo) under-represents that case for a mapping net trained
    from scratch without the repo's additional real-old-photo VAE_A domain
    adaptation -- raise this for training runs targeting B&W restoration."""
    B = Image.open(clean_path).convert("RGB")
    A = online_add_degradation_v2(B)
    if A.size != B.size:
        A = A.resize(B.size, Image.BILINEAR)

    if random.uniform(0, 1) < grayscale_prob:
        A = A.convert("L").convert("RGB")
        B = B.convert("L").convert("RGB")

    pos, flip = _crop_params(B.size)
    A = _apply_crop_flip(A, pos, flip)
    B = _apply_crop_flip(B, pos, flip)

    if A.size != (FINE_SIZE, FINE_SIZE):
        A = A.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)
        B = B.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)

    return _to_tensor_norm(A), _to_tensor_norm(B)


class OnlineQualityDataset(Dataset):
    """len() is nominal (controls how many samples count as "one epoch" for
    DataLoader bookkeeping / shuffling), but __getitem__ always draws a fresh
    random source image + fresh degradation, regardless of idx. This is the
    key difference from a dataset that reads a fixed PNG from disk per idx."""

    def __init__(self, clean_dir, length=20000, min_side=256, grayscale_prob=0.1):
        self.files = list_clean_images(clean_dir, min_side)
        assert self.files, f"no usable images (>= {min_side}px both sides) in {clean_dir}"
        self.length = length
        self.grayscale_prob = grayscale_prob

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        for _ in range(5):  # a handful of retries in case of a corrupt/transient-unreadable file
            f = random.choice(self.files)
            try:
                return make_online_pair(f, self.grayscale_prob)
            except Exception as e:
                print(f"[OnlineQualityDataset] skipping unreadable {f}: {e}")
        raise RuntimeError("too many consecutive unreadable files -- check your image folder")


def make_online_train_loader(clean_dir, batch_size=16, epoch_length=20000, num_workers=4, grayscale_prob=0.1):
    """Online training loader: every sample is freshly degraded. Use a
    multi-worker DataLoader since degradation (PIL/cv2 ops) is CPU-bound and
    would otherwise bottleneck the GPU."""
    ds = OnlineQualityDataset(clean_dir, length=epoch_length, grayscale_prob=grayscale_prob)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )
    print(f"online train loader: {len(ds.files)} source images, "
          f"{epoch_length} samples/epoch, {len(loader)} batches/epoch")
    return loader


# --------------------------------------------------------------------------- #
# Train/val split at the SOURCE-IMAGE level, both drawn from the same folder.
# --------------------------------------------------------------------------- #
class FixedPairDataset(Dataset):
    """A small, pre-generated, in-memory set of (A, B) tensor pairs. Built
    once with a fixed seed so it's identical across runs/evals, but drawn
    from source images that were NEVER used for training."""

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def make_online_dataloaders(clean_dir, batch_size=16, epoch_length=20000,
                             num_workers=2, val_fraction=0.05, val_samples=400, seed=0,
                             grayscale_prob=0.1):
    """Splits source images into train/val pools (no overlap), builds the
    online training loader from the train pool, and a FIXED val set (built
    once, with a fixed seed, same degradation pipeline) from the held-out
    val pool -- so val is comparable across runs AND actually measures
    generalization on the same visual domain the model is training on."""
    all_files = list_clean_images(clean_dir)
    rng = random.Random(seed)
    shuffled = all_files[:]
    rng.shuffle(shuffled)
    n_val_files = max(1, int(len(shuffled) * val_fraction))
    val_files, train_files = shuffled[:n_val_files], shuffled[n_val_files:]
    print(f"split: {len(train_files)} train source images, {len(val_files)} val source images (no overlap)")

    train_ds = OnlineQualityDataset.__new__(OnlineQualityDataset)  # bypass list_clean_images re-scan
    train_ds.files = train_files
    train_ds.length = epoch_length
    train_ds.grayscale_prob = grayscale_prob
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )

    # build the fixed val set once, with a local seeded RNG so it's reproducible
    state = random.getstate()
    random.seed(seed + 1)
    pairs = []
    i = 0
    while len(pairs) < val_samples:
        f = val_files[i % len(val_files)]
        try:
            pairs.append(make_online_pair(f, grayscale_prob))
        except Exception as e:
            print(f"[val set] skipping unreadable {f}: {e}")
        i += 1
    random.setstate(state)

    val_loader = DataLoader(FixedPairDataset(pairs), batch_size=batch_size, shuffle=False, num_workers=0)
    print(f"online train loader: {len(train_files)} source images, {epoch_length} samples/epoch, "
          f"{len(train_loader)} batches/epoch | fixed val: {len(pairs)} samples from {len(val_files)} held-out images")
    return train_loader, val_loader

### `benchmark.py`

In [ ]:
%%writefile /content/benchmark.py
"""
Large, independent, frozen benchmark for the old-photo mapping.

build_benchmark(): takes clean photos the models never saw (e.g. COCO val2017 or
DIV2K valid), makes degraded/clean 256x256 pairs with the SAME synthetic
degradation as training (make_online_pair: online_add_degradation_v2, crop,
50% gray), each pair seeded by its index, and saves them as PNGs + meta.csv.
Built once (resumable), so every model is scored on exactly the same pairs.

evaluate_benchmark(): streams the pairs from disk and reports, per method,
PSNR, SSIM, LPIPS, FID, KID (unbiased at any n; mean +- std over subsets),
CIEDE2000 colour error, a gray / colour split, and mapping cost. Reference rows:
degraded input, Microsoft mapping, and the VAE B ceiling (clean -> VAE B -> clean,
the best any mapping can do).

FID / KID are computed between each method's outputs and the PAIRED clean targets
(same scenes). FID values are therefore only comparable between methods on the
same benchmark, never with other papers or other n; KID is unbiased, so its
differences can be judged against its std. (A clean-vs-clean comparison of two
disjoint halves is NOT a valid floor here: it measures scene differences too,
which is why it can exceed the ceiling's FID.)
"""
import csv
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from skimage.metrics import structural_similarity

from online_dataset import list_clean_images, make_online_pair


# --------------------------------------------------------------------------- #
# Building the benchmark
# --------------------------------------------------------------------------- #
def _to_uint8(t):
    """(3,H,W) in [-1,1] -> (H,W,3) uint8."""
    return ((t.clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).numpy()


def build_benchmark(src_dir, out_dir, n=5000, crops_per_image=1, seed=0, grayscale_prob=0.5):
    """Saves n degraded/clean pairs: out_dir/A/000000.png, out_dir/B/000000.png, meta.csv.
    Resumable: pairs already listed in meta.csv are skipped. Pair i uses source image
    i // crops_per_image (of a seeded shuffle) and its own fixed random seed."""
    os.makedirs(f"{out_dir}/A", exist_ok=True)
    os.makedirs(f"{out_dir}/B", exist_ok=True)
    meta_path = f"{out_dir}/meta.csv"
    files = list_clean_images(src_dir)
    random.Random(seed).shuffle(files)
    total = min(n, len(files) * crops_per_image)
    done = set()
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            done = {int(r["idx"]) for r in csv.DictReader(f)}
    print(f"benchmark: {total} pairs from {len(files)} source images ({len(done)} already built)")
    new = not os.path.exists(meta_path)
    with open(meta_path, "a", newline="") as mf:
        w = csv.writer(mf)
        if new:
            w.writerow(["idx", "source", "gray"])
        t0, made = time.time(), 0
        for i in range(total):
            if i in done:
                continue
            src = files[i // crops_per_image]
            s = seed * 1_000_003 + i
            random.seed(s)
            np.random.seed(s % 2 ** 32)
            try:
                a, b = make_online_pair(src, grayscale_prob)
            except Exception as e:
                print(f"  skipping {src}: {e}")
                continue
            gray = bool((b - b.mean(0, keepdim=True)).abs().max() < 1e-3)
            Image.fromarray(_to_uint8(a)).save(f"{out_dir}/A/{i:06d}.png")
            Image.fromarray(_to_uint8(b)).save(f"{out_dir}/B/{i:06d}.png")
            w.writerow([i, os.path.basename(src), int(gray)])
            made += 1
            if made % 250 == 0:
                mf.flush()
                print(f"  {len(done) + made}/{total} pairs ({made / (time.time() - t0):.1f}/s)")
    print(f"benchmark ready: {out_dir}")


class BenchPairs(torch.utils.data.Dataset):
    """(A, B, gray) from a built benchmark; A, B in [-1,1]."""

    def __init__(self, bench_dir, max_images=None):
        self.dir = bench_dir
        with open(f"{bench_dir}/meta.csv") as f:
            rows = sorted(csv.DictReader(f), key=lambda r: int(r["idx"]))
        self.rows = rows[:max_images] if max_images else rows

    def __len__(self):
        return len(self.rows)

    def _load(self, sub, idx):
        a = np.asarray(Image.open(f"{self.dir}/{sub}/{idx:06d}.png").convert("RGB"), dtype=np.float32)
        return torch.from_numpy(a / 127.5 - 1.0).permute(2, 0, 1)

    def __getitem__(self, k):
        idx = int(self.rows[k]["idx"])
        return self._load("A", idx), self._load("B", idx), int(self.rows[k]["gray"])


# --------------------------------------------------------------------------- #
# Colour science (same code as color_analysis.py, checked against scikit-image)
# --------------------------------------------------------------------------- #
_M_RGB2XYZ = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                           [0.2126729, 0.7151522, 0.0721750],
                           [0.0193339, 0.1191920, 0.9503041]])
_WHITE_D65 = torch.tensor([0.95047, 1.0, 1.08883])


def srgb_to_lab(rgb):
    """(N,3,H,W) sRGB in [0,1] -> (N,3,H,W) CIELAB (D65, 2 deg), like skimage.color.rgb2lab."""
    rgb = rgb.clamp(0, 1)
    lin = torch.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    xyz = torch.einsum("ij,njhw->nihw", _M_RGB2XYZ.to(rgb), lin) / _WHITE_D65.to(rgb)[None, :, None, None]
    eps = (6 / 29) ** 3
    f = torch.where(xyz > eps, xyz.clamp_min(1e-12) ** (1 / 3), xyz / (3 * (6 / 29) ** 2) + 4 / 29)
    L = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return torch.stack([L, a, b], 1)


def ciede2000(lab1, lab2):
    """Per-pixel CIEDE2000 between (N,3,H,W) Lab tensors -> (N,H,W). (Sharma et al. 2005)"""
    L1, a1, b1 = lab1[:, 0], lab1[:, 1], lab1[:, 2]
    L2, a2, b2 = lab2[:, 0], lab2[:, 1], lab2[:, 2]
    C1 = torch.sqrt(a1 ** 2 + b1 ** 2)
    C2 = torch.sqrt(a2 ** 2 + b2 ** 2)
    Cb7 = ((C1 + C2) / 2) ** 7
    G = 0.5 * (1 - torch.sqrt(Cb7 / (Cb7 + 25.0 ** 7)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p = torch.sqrt(a1p ** 2 + b1 ** 2)
    C2p = torch.sqrt(a2p ** 2 + b2 ** 2)
    h1p = torch.rad2deg(torch.atan2(b1, a1p)) % 360
    h2p = torch.rad2deg(torch.atan2(b2, a2p)) % 360
    zero = (C1p * C2p) == 0

    dLp = L2 - L1
    dCp = C2p - C1p
    dh = h2p - h1p
    dh = torch.where(dh > 180, dh - 360, torch.where(dh < -180, dh + 360, dh))
    dh = torch.where(zero, torch.zeros_like(dh), dh)
    dHp = 2 * torch.sqrt(C1p * C2p) * torch.sin(torch.deg2rad(dh / 2))

    Lbp = (L1 + L2) / 2
    Cbp = (C1p + C2p) / 2
    hsum = h1p + h2p
    hbp = torch.where((h1p - h2p).abs() <= 180, hsum / 2,
                      torch.where(hsum < 360, (hsum + 360) / 2, (hsum - 360) / 2))
    hbp = torch.where(zero, hsum, hbp)
    T = (1 - 0.17 * torch.cos(torch.deg2rad(hbp - 30)) + 0.24 * torch.cos(torch.deg2rad(2 * hbp))
         + 0.32 * torch.cos(torch.deg2rad(3 * hbp + 6)) - 0.20 * torch.cos(torch.deg2rad(4 * hbp - 63)))
    dtheta = 30 * torch.exp(-(((hbp - 275) / 25) ** 2))
    Cbp7 = Cbp ** 7
    RC = 2 * torch.sqrt(Cbp7 / (Cbp7 + 25.0 ** 7))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / torch.sqrt(20 + (Lbp - 50) ** 2)
    SC = 1 + 0.045 * Cbp
    SH = 1 + 0.015 * Cbp * T
    RT = -torch.sin(torch.deg2rad(2 * dtheta)) * RC
    return torch.sqrt(((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2
                       + RT * (dCp / SC) * (dHp / SH)).clamp_min(0))


# --------------------------------------------------------------------------- #
# Distribution metrics
# --------------------------------------------------------------------------- #
def fid_from_feats(f1, f2):
    from pytorch_fid.fid_score import calculate_frechet_distance
    f1, f2 = f1.astype(np.float64), f2.astype(np.float64)
    return float(calculate_frechet_distance(f1.mean(0), np.cov(f1, rowvar=False),
                                            f2.mean(0), np.cov(f2, rowvar=False)))


def kid_from_feats(f1, f2, n_subsets=100, subset_size=1000, seed=0):
    """Kernel Inception Distance (Binkowski et al. 2018): unbiased MMD^2 with the
    cubic polynomial kernel k(x,y) = (x.y/d + 1)^3, averaged over random subsets.
    Returns (mean, std). Unlike FID it is unbiased, so it is comparable across n."""
    rng = np.random.default_rng(seed)
    d = f1.shape[1]
    m = min(subset_size, len(f1), len(f2))
    vals = []
    for _ in range(n_subsets):
        x = f1[rng.choice(len(f1), m, replace=False)].astype(np.float64)
        y = f2[rng.choice(len(f2), m, replace=False)].astype(np.float64)
        kxx = (x @ x.T / d + 1) ** 3
        kyy = (y @ y.T / d + 1) ** 3
        kxy = (x @ y.T / d + 1) ** 3
        mmd = ((kxx.sum() - np.trace(kxx)) + (kyy.sum() - np.trace(kyy))) / (m * (m - 1)) - 2 * kxy.mean()
        vals.append(mmd)
    return float(np.mean(vals)), float(np.std(vals))


# --------------------------------------------------------------------------- #
# Evaluation
# --------------------------------------------------------------------------- #
class _Metrics:
    def __init__(self, device):
        import lpips
        from pytorch_fid.inception import InceptionV3
        self.lpips = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
        self.inception = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()

    @torch.no_grad()
    def per_image(self, y, t):
        """y, t: (N,3,H,W) in [0,1] -> dict of per-image metrics (CPU)."""
        mse = ((y - t) ** 2).flatten(1).mean(1)
        p = y.permute(0, 2, 3, 1).cpu().numpy()
        q = t.permute(0, 2, 3, 1).cpu().numpy()
        ssim = [structural_similarity(q[k], p[k], channel_axis=2, data_range=1.0, gaussian_weights=True,
                                      sigma=1.5, use_sample_covariance=False) for k in range(len(p))]
        return {"psnr": (10 * torch.log10(1.0 / mse.clamp_min(1e-10))).cpu(),
                "ssim": torch.tensor(ssim, dtype=torch.float32),
                "lpips": self.lpips(y * 2 - 1, t * 2 - 1).flatten().cpu(),
                "de00": ciede2000(srgb_to_lab(t), srgb_to_lab(y)).flatten(1).mean(1).cpu()}

    @torch.no_grad()
    def feats(self, y):
        return self.inception(y)[0].flatten(1).cpu().numpy()


@torch.no_grad()
def evaluate_benchmark(obj, rfs, bench_dir, rf_steps=(1,), batch_size=16, max_images=None,
                       include_input=True, include_ceiling=True, out_csv=None, num_workers=2,
                       include_microsoft=True):
    """obj: OldPhotoModel (quality); rfs: {"name": RFMapping}. Bridge noise is seeded per method."""
    device = obj.device
    metrics = _Metrics(device)
    ds = BenchPairs(bench_dir, max_images)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    print(f"evaluating on {len(ds)} pairs from {bench_dir}")

    methods = []
    if include_input:
        methods.append(("degraded input", "input", None, None))
    if include_microsoft:
        methods.append(("Microsoft mapping", "net", obj._core.mapping_net, None))
    for name, rf in rfs.items():
        for s in rf_steps:            # s=None: a direct mapping (no sampler), labelled by its name only
            methods.append((name if s is None else f"{name} ({s}-step)", "net", rf, s))
    if include_ceiling:
        methods.append(("VAE B ceiling", "ceiling", None, None))

    to01 = lambda x: ((x.float() + 1) / 2).clamp(0, 1)
    real_feats, gray_all, rows = None, None, []
    for name, kind, net, steps in methods:
        per = {k: [] for k in ("psnr", "ssim", "lpips", "de00")}
        feats, real, grays, t_map = [], [], [], 0.0
        torch.manual_seed(0)
        for A, B, g in loader:
            A, B = A.to(device), B.to(device)
            if kind == "input":
                y = to01(A)
            elif kind == "ceiling":
                y = to01(obj.decode_B(obj.encode_B(B)))
            else:
                if steps is not None:
                    net.steps = steps
                z_A = obj.encode_A(A)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                t0 = time.time()
                z_B = net(z_A)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                t_map += time.time() - t0
                y = to01(obj.decode_B(z_B))
            t = to01(B)
            for k, v in metrics.per_image(y, t).items():
                per[k].append(v)
            feats.append(metrics.feats(y))
            if real_feats is None:
                real.append(metrics.feats(t))
                grays.append(g.bool())
        if real_feats is None:
            real_feats, gray_all = np.concatenate(real), torch.cat(grays)
        per = {k: torch.cat(v) for k, v in per.items()}
        f = np.concatenate(feats)
        kid_m, kid_s = kid_from_feats(f, real_feats)
        row = {"method": name, "PSNR↑": per["psnr"].mean().item(), "SSIM↑": per["ssim"].mean().item(),
               "LPIPS↓": per["lpips"].mean().item(), "FID↓": fid_from_feats(f, real_feats),
               "KID×1000↓": 1000 * kid_m, "KID std×1000": 1000 * kid_s, "ΔE00↓": per["de00"].mean().item()}
        for tag, m in (("gray", gray_all), ("colour", ~gray_all)):
            if m.any():
                for k, col in (("psnr", "PSNR"), ("ssim", "SSIM"), ("lpips", "LPIPS"), ("de00", "ΔE00")):
                    row[f"{col} {tag}"] = per[k][m].mean().item()
        if kind == "net":
            row["mapping params (M)"] = sum(p.numel() for p in net.parameters()) / 1e6
            row["mapping ms/img"] = 1000 * t_map / len(ds)
        rows.append(row)
        print(f"{name:34s} PSNR {row['PSNR↑']:.2f}  SSIM {row['SSIM↑']:.4f}  LPIPS {row['LPIPS↓']:.4f}  "
              f"FID {row['FID↓']:.2f}  KID×1000 {row['KID×1000↓']:.2f}±{row['KID std×1000']:.2f}  ΔE00 {row['ΔE00↓']:.2f}")

    df = pd.DataFrame(rows).set_index("method")
    if out_csv:
        df.to_csv(out_csv)
        print(f"saved {out_csv}")
    return df.round(4)

## Download COCO val2017 and build the benchmark

In [ ]:
ROOT = "/content/drive/MyDrive/lightweight_old_photos"      # project folder on Google Drive
BENCH_DIR = f"{ROOT}/benchmarks/coco_5000"
SRC_DIR = "/content/coco/val2017"

if not os.path.isdir(SRC_DIR):
    os.makedirs("/content/coco", exist_ok=True)
    !wget -q -c http://images.cocodataset.org/zips/val2017.zip -O /content/coco/val2017.zip
    !unzip -q -n /content/coco/val2017.zip -d /content/coco

from benchmark import build_benchmark
build_benchmark(SRC_DIR, BENCH_DIR, n=5000, crops_per_image=1, seed=0, grayscale_prob=0.5)

## Score the reference rows

In [ ]:
from benchmark import evaluate_benchmark
from old_photo_model import OldPhotoModel

obj = OldPhotoModel(mode="quality", gpu=0)
os.makedirs(f"{ROOT}/results", exist_ok=True)
ref = evaluate_benchmark(obj, {}, BENCH_DIR, include_input=True, include_ceiling=True,
                         out_csv=f"{ROOT}/results/coco_references.csv")
ref